In [ ]:
import pandas as pd, numpy as np
import torch
from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cpu")
MIXED_PRECISION = False

Mounted at /content/drive


In [ ]:
len(dataset)

96199

In [ ]:
# Save DataFrame to CSV
dataset.to_csv("cleaned_blog_authorship.csv", index=False)

# Download to your computer
from google.colab import files
files.download("cleaned_blog_authorship.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
DATA_PATH = "/content/drive/MyDrive/blogtext.csv"

df = pd.read_csv(DATA_PATH)
print("Rows:", len(df))
print("Columns:", list(df.columns))

Rows: 681284
Columns: ['id', 'gender', 'age', 'topic', 'sign', 'date', 'text']


In [ ]:
df = df[df["topic"].str.lower() != "indunk"]
print(len(df))

430269


In [ ]:
df["word_count"] = df["text"].apply(lambda x: len(str(x).split()))
df = df[df["word_count"] >= 200]

In [ ]:
len(df)

142014

In [ ]:
!pip install langid
import langid
df = df[df["text"].apply(lambda x: langid.classify(str(x))[0] == "en")]

print(len(df))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 13.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langid: filename=langid-1.1.6-py3-none-any.whl size=1941171 sha256=92b40a78932df28c6ad6faa409ec5d211e7188963d7a670106b8b388442a908c
  Stored in directory: /root/.cache/pip/wheels/3c/bc/9d/266e27289b9019680d65d9b608c37bff1eff565b001c977ec5
Successfully built langid
141213


In [ ]:
df.head()

,id,gender,age,topic,sign,date,text,word_count
2,2059027,male,15,Student,Leo,"12,May,2004",In het kader van kernfusie op aarde...,4326
5,3581210,male,33,InvestmentBanking,Aquarius,"10,June,2004",I had an interesting conversation...,662
7,3581210,male,33,InvestmentBanking,Aquarius,"10,June,2004","If anything, Korea is a country o...",387
8,3581210,male,33,InvestmentBanking,Aquarius,"10,June,2004",Take a read of this news article ...,386
10,3581210,male,33,InvestmentBanking,Aquarius,"09,June,2004","Ah, the Korean language...it look...",296


In [ ]:
# def bucket_age(a):
#     try:
#         a = int(a)
#     except:
#         return "unknown"
#     if 13 <= a <= 17: return "10s"
#     if 23 <= a <= 27: return "20s"
#     if 33 <= a <= 47: return "30s"
#     return "unknown"

# df["age_bucket"] = df["age"].apply(bucket_age).astype(str)
# df = df[df["age_bucket"] != "unknown"]

# df["word_count"] = df["text"].apply(lambda x: len(str(x).split()))

In [ ]:
merge_map = {
    # student
    "Student": "Student",
    # singletons
    "Technology":"Technology", "Education":"Education", "Arts":"Arts",
    "Communications-Media":"Communications-Media", "Internet":"Internet",
    "Non-Profit":"Non-Profit",
    # science & technical
    "Engineering":"Science & Technical", "Science":"Science & Technical",
    "Biotech":"Science & Technical", "Telecommunications":"Science & Technical",
    # creative media & culture
    "Publishing":"Creative Media & Culture", "Fashion":"Creative Media & Culture",
    "Museums-Libraries":"Creative Media & Culture", "Advertising":"Creative Media & Culture",
    "Marketing":"Creative Media & Culture",
    # business & consulting
    "Consulting":"Business & Consulting", "BusinessServices":"Business & Consulting",
    "Accounting":"Business & Consulting", "HumanResources":"Business & Consulting",
    # finance & property
    "Banking":"Finance & Property", "RealEstate":"Finance & Property",
    "InvestmentBanking":"Finance & Property",
    # public service & governance
    "Government":"Public Service & Governance", "Military":"Public Service & Governance",
    "LawEnforcement-Security":"Public Service & Governance",
    # law & specialized
    "Law":"Law & Specialized Services", "Religion":"Law & Specialized Services",
    # industrial & misc.
    "Chemicals":"Industrial & Misc.", "Sports-Recreation":"Industrial & Misc.",
    "Transportation":"Industrial & Misc.", "Manufacturing":"Industrial & Misc.",
    "Tourism":"Industrial & Misc.", "Architecture":"Industrial & Misc.",
    "Automotive":"Industrial & Misc.", "Agriculture":"Industrial & Misc.",
    "Construction":"Industrial & Misc.", "Environment":"Industrial & Misc.",
    "Maritime":"Industrial & Misc.",
}

df["industry"] = df["topic"].map(merge_map)

In [ ]:
df.sample(5, random_state=44)

,id,gender,age,topic,sign,date,text,word_count,industry
29998,3310475,male,16,Biotech,Gemini,"10,June,2004",a tiring day after both English and ...,380,Science & Technical
583900,1637100,female,16,Student,Scorpio,"26,July,2004",*sighz* why am i a class-1 bitch? ...,281,Student
466609,3483026,female,13,Arts,Taurus,"05,July,2004",Not until now have I noticed how si...,420,Arts
293581,3140401,male,24,Internet,Capricorn,"24,June,2004",#1: Where am I going to find a ment...,486,Internet
163870,3500491,female,25,Arts,Libra,"31,May,2004",There's a John Denver song that I alway...,547,Arts


In [ ]:
len(df)

141213

In [ ]:
import re

def clean_noise(text):
    if not text:
        return ""
    text = str(text)
    # Remove *all* tags <...> regardless of what’s inside
    text = re.sub(r"<[^>]*>", " ", text)
    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Apply to DataFrame
df["text"] = df["text"].apply(clean_noise)

In [ ]:
import re
from html import unescape

def clean_noise(text):
    if not text:
        return ""
    text = str(text)
    # Decode HTML entities (&nbsp; → space, &amp; → &)
    text = unescape(text)
    # Remove all tags <...>
    text = re.sub(r"<[^>]*>", " ", text)
    # Remove any leftover entities like &nbsp; or &#39;
    text = re.sub(r"&\w+;|&#\d+;", " ", text)
    # Collapse spaces
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["text"] = df["text"].apply(clean_noise)

# Now check again
print(df["text"].str.contains("nbsp", case=False).any())

False


In [ ]:
import re

# Updated regex patterns
URL_RE = re.compile(r'\b(?:https?://|www\.|urllink)\S*\b', re.IGNORECASE)
EMAIL_RE = re.compile(r'\b[\w.+-]+@[\w-]+\.[\w.-]+\b')
USERNAME_RE = re.compile(r'@\w[\w.-]*\b')
HASHTAG_RE = re.compile(r'#[\w\d_]+')

def replace_specials(text: str) -> str:
    if not text:
        return ""
    # Replace in specific order to avoid conflicts
    text = URL_RE.sub(" <url> ", text.lower().replace("urllink", "<url>"))
    text = EMAIL_RE.sub(" <email> ", text)
    text = USERNAME_RE.sub(" <user> ", text)
    text = HASHTAG_RE.sub(" <hashtag> ", text)
    return text.strip()

In [ ]:
# apply cleaning function to all posts
# df["text"] = df["text"].map(replace_specials)

# # quick check
df.sample(5, random_state=42)

,id,gender,age,topic,sign,date,text,word_count,industry
82584,3892901,male,27,Arts,Aquarius,"15,July,2004",my instructor told us today he wanted us to ke...,477,Arts
74978,3228371,male,13,Student,Virgo,"09,June,2004",just been to band practise. very interesting. ...,256,Student
30397,3458177,female,24,Banking,Aries,"01,August,2004",i saw mop yesterday...oooh.. maybe i shd enlig...,292,Finance & Property
248033,4075390,female,24,Non-Profit,Pisces,"21,May,2004",so after a kickin' start with the stills and c...,207,Non-Profit
386112,2834784,male,17,Biotech,Libra,"14,February,2004",i had very strange dreams over the weekend. an...,584,Science & Technical


In [ ]:
count_email = df["text"].str.count("<url>").sum()
print("Total <email> placeholders:", count_email)

Total <email> placeholders: 99144


In [ ]:
len(df)
df.to_csv("/content/drive/MyDrive/final_data.csv")

In [ ]:
len(df)

141213

In [ ]:
import re

AGE_PATS_EXT = [
    r"\bI\s*(?:'m|am)\s*\d{1,2}\b",                 # I'm 19 / I am 19
    r"\b\d{1,2}\s*(?:years?|yrs?)\s*old\b",         # 21 years old / 21 yrs old
    r"\b\d{1,2}\s*(?:y/?o|yo)\b",                   # 21 y/o, 21 yo
    r"\b\d{1,2}[-\s]?(?:year|yr)s?[-\s]?old\b",     # 21-year-old / 21 yr-old
    r"\bAge(?:d)?[:=]?\s*\d{1,2}\b",                # Age: 17 / Aged 17 / Age=17
    r"\bjust\s+turned\s+\d{1,2}\b",                 # just turned 21
    r"\bturned\s+\d{1,2}\b",                        # turned 18
    r"\bturning\s+\d{1,2}\b",                       # turning 20
    r"\b(?:my|his|her)\s+\d{1,2}(?:st|nd|rd|th)\s+birthday\b"  # my 18th birthday
]

def clean_age_mentions_extended(text):
    s = str(text)
    for p in AGE_PATS_EXT:
        s = re.sub(p, "<AGE>", s, flags=re.IGNORECASE)
    return s

df["text"] = df["text"].apply(clean_age_mentions_extended)

In [ ]:
# Count total <AGE> placeholders across all texts
count_age = df["text"].str.count("<AGE>").sum()
print("Total <AGE> placeholders:", count_age)

# Count how many rows contain at least one <AGE>
rows_with_age = df["text"].str.contains("<AGE>").sum()
print("Rows with <AGE>:", rows_with_age)

Total <AGE> placeholders: 10722
Rows with <AGE>: 8326


In [ ]:
# ----- SAVED SO FAR
df.to_csv("/content/drive/MyDrive/final_data.csv")

In [ ]:
import re

phrases = [
    "posted by",
    "leave a comment",
    "subscribe to",
    "click here",
    "continue reading",
    "all rights reserved",
    "read more",
    "copyright",
    "follow me on",
    "add to favorites"
]

# build one regex
pattern = re.compile("|".join([re.escape(p) for p in phrases]), flags=re.IGNORECASE)

# remove phrases wherever they appear
df["text"] = df["text"].str.replace(pattern, "", regex=True)

In [ ]:
import re
import pandas as pd

def normalize_elongation(text, keep_letters=3, keep_punct=3, mark_token="<ELONG>"):
    text = str(text)
    # Letters (e.g., loooove → loo<ELONG>ve, case-insensitive)
    text = re.sub(r"(\w)\1{2,}(\w*)", lambda m: m.group(1) * keep_letters + mark_token + m.group(2), text, flags=re.IGNORECASE)
    # Punctuation and emojis (e.g., !!!! → !!!<ELONG>, 😊😊😊 → 😊😊😊<ELONG>)
    text = re.sub(r"([!?\.,\-_\~\*\^\:#;😀-🙏])\1{2,}", lambda m: m.group(1) * keep_punct + mark_token, text)
    return text

# Apply to DataFrame
df["text"] = df["text"].astype("string").fillna("").map(normalize_elongation)

In [ ]:
# Count total <AGE> placeholders across all texts
count_age = df["text"].str.count("<ELONG>").sum()
print("Total <AGE> placeholders:", count_age)

# Count how many rows contain at least one <AGE>
rows_with_age = df["text"].str.contains("<ELONG>").sum()
print("Rows with <AGE>:", rows_with_age)

Total <AGE> placeholders: 678065
Rows with <AGE>: 96946


In [ ]:
# ------------------------ SAVED
df.to_csv("/content/drive/MyDrive/final_data.csv")

In [ ]:
len(df)

141213

In [ ]:
df.head()

,id,gender,age,topic,sign,date,text,word_count,industry
2,2059027,male,15,Student,Leo,"12,May,2004",in het kader van kernfusie op aarde: maak je e...,4326,Student
5,3581210,male,33,InvestmentBanking,Aquarius,"10,June,2004",i had an interesting conversation with my dad ...,662,Finance & Property
7,3581210,male,33,InvestmentBanking,Aquarius,"10,June,2004","if anything, korea is a country of extremes. e...",387,Finance & Property
8,3581210,male,33,InvestmentBanking,Aquarius,"10,June,2004",take a read of this news article from <url> jo...,386,Finance & Property
10,3581210,male,33,InvestmentBanking,Aquarius,"09,June,2004","ah, the korean language...<ELONG>it looks so d...",296,Finance & Property


In [ ]:
!pip install langid
import langid
df = df[df["text"].apply(lambda x: langid.classify(str(x))[0] == "en")]

print(len(df))

141195


In [ ]:
len(df)

141195

In [ ]:
# ------------------------ final data that includes all 200-1500 including students - all clean data with long text extracttion, we should only add students that should be cleaned
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/all_clean_data_without_students.csv")
students = pd.read_csv("/content/drive/MyDrive/students.csv")

In [ ]:
len(df)

92727

In [ ]:
students.head()

,Unnamed: 0,id,gender,age,topic,sign,date,text,word_count,industry
0,2,2059027,male,15,Student,Leo,"12,May,2004",in het kader van kernfusie op aarde: maak je e...,4326,Student
1,98,3668238,female,17,Student,Gemini,"26,June,2004","it was fun :) hey dad i’m writing to you, not ...",450,Student
2,99,3668238,female,17,Student,Gemini,"24,June,2004",war is every why are we so fixated with it?? i...,264,Student
3,100,3668238,female,17,Student,Gemini,"23,June,2004","i can't wait for things to get moving, i want ...",546,Student
4,101,3668238,female,17,Student,Gemini,"20,June,2004",i wrote this out all interestin n happy n then...,475,Student


In [ ]:
print("len: ", len(students))
industry_counts = students["industry"].value_counts()
print(industry_counts)

len:  50030
industry
Student    50030
Name: count, dtype: int64


In [ ]:
print("len: ", len(df))
industry_counts = df["industry"].value_counts()
print(industry_counts)

len:  92727
industry
Technology                     10955
Education                      10944
Arts                           10596
Creative Media & Culture        8801
Science & Technical             8179
Communications-Media            7151
Business & Consulting           6725
Industrial & Misc.              6549
Law & Specialized Services      5523
Non-Profit                      5029
Internet                        4849
Public Service & Governance     4450
Finance & Property              2976
Name: count, dtype: int64


In [ ]:
df = df[["id", "gender", "age", "industry", "text", "word_count"]]
print(len(students))

50030


In [ ]:
students = students[students["word_count"] <= 1500]

In [ ]:
len(students)

49193

In [ ]:
import re

# terms = [
#     r"university|college|school|campus|dorm|classroom|mentor",
#     r"student(?:s)?|principal",
#     r"professor(?:s)?|teacher(?:s)?|instructor(?:s)?",
#     r"class(?:es)?", r"classroom", r"seminar(?:s)?",
#     r"teacher(?:s)?", r"professor(?:s)?", r"lecturer(?:s)?", r"instructor(?:s)?",
#     r"advisor(?:s)?", r"mentor(?:s)?", r"tutor(?:s)?", r"dean(?:s)?", r"principal(?:s)?",
#     r"class(?:es)?|lecture(?:s)?|seminar(?:s)?",
#     r"exam(?:s)?|quiz(?:zes)?|midterm(?:s)?|final(?:s)?",
#     r"homework|assignment(?:s)?|coursework",
#     r"project(?:s)?|presentation(?:s)?",
#     r"grade(?:s)?|gpa|curriculum|principal|classmate(?:s)",
#     r"semester(?:s)?|term(?:s)?",
#     r"campus|dorm(?:itory)?",
#     r"lab(?:s)?|laborator(?:y|ies)",
#     r"library",
#     r"syllabus",
#     r"advisor(?:s)?|mentor(?:s)?",
#     r"thesis|dissertation|paper(?:s)?|essay(?:s)?",
#     r"diploma", r"degree(?:s)?", r"phd|doctorate",
#     r"scholarship(?:s)?|internship(?:s)?|credit(?:s)?"
# ]

terms = [
    r"university", r"college", r"school", r"campus", r"dorm(?:itory)?", r"study(?:ing)", r"apply(?:ing)", r"try(?:ing)",
    r"class(?:es)?", r"classroom", r"lecture(?:s)?", r"seminar(?:s)?", r"tutorial(?:s)?", r"workshop(?:s)?",
    r"teacher(?:s)?", r"professor(?:s)?", r"lecturer(?:s)?", r"instructor(?:s)?", r"learn(?:ing)", r"practice",
    r"advisor(?:s)?", r"mentor(?:s)?", r"tutor(?:s)?", r"dean(?:s)?", r"principal(?:s)?", r"cheat(?:ing)",
    r"student(?:s)?", r"classmate(?:s)?", r"roommate(?:s)?", r"peer(?:s)?", r"work(?:ing)",
    r"homework", r"assignment(?:s)?", r"project(?:s)?", r"group project(?:s)?",
    r"essay(?:s)?", r"paper(?:s)?", r"report(?:s)?", r"thesis", r"dissertation", r"academic",
    r"exam(?:s)?", r"test(?:s)?", r"quiz(?:zes)?", r"midterm(?:s)?", r"final(?:s)?", r"responsible",
    r"grade(?:s)?", r"GPA", r"credit(?:s)?", r"apply", r"study", r"smart", r"intelligent", r"clever",
    r"syllabus", r"curriculum", r"timetable", r"schedule", r"deadline", r"canteen",
    r"semester(?:s)?", r"trimester(?:s)?", r"quarter(?:s)?", r"academic year",
    r"library", r"textbook(?:s)?", r"laborator(?:y|ies)", r"lab(?:s)?", r"discuss(?:sion)", r"translate",
    r"presentation(?:s)?", r"slides?", r"research", r"academy", r"deadline",
    r"scholarship(?:s)?", r"fellowship(?:s)?", r"internship(?:s)?",
    r"diploma", r"degree(?:s)?", r"bachelor'?s?", r"master'?s?", r"phd|doctorate",
    r"office hours", r"report card", r"ambiguous", r"motivated"
]

pattern = re.compile(r"\b(?:%s)\b" % "|".join(terms), re.IGNORECASE)

def count_keywords(text: str) -> int:
    return len({m.group(0).lower() for m in pattern.finditer(str(text))})

students["keyword_count"] = students["text"].apply(count_keywords)
student_related = students[students["keyword_count"] >= 3]
print("Extracted rows:", len(student_related))

# terms = [
#     r"university", r"college", r"school", r"campus", r"dorm(?:itory)?",
#     r"class(?:es)?", r"classroom", r"lecture(?:s)?", r"seminar(?:s)?", r"tutorial(?:s)?", r"workshop(?:s)?",
#     r"teacher(?:s)?", r"professor(?:s)?", r"lecturer(?:s)?", r"instructor(?:s)?",
#     r"advisor(?:s)?", r"mentor(?:s)?", r"tutor(?:s)?", r"dean(?:s)?", r"principal(?:s)?",
#     r"student(?:s)?", r"classmate(?:s)?", r"roommate(?:s)?", r"peer(?:s)?",
#     r"homework", r"assignment(?:s)?", r"project(?:s)?", r"group project(?:s)?",
#     r"essay(?:s)?", r"paper(?:s)?", r"report(?:s)?", r"thesis", r"dissertation",
#     r"exam(?:s)?", r"test(?:s)?", r"quiz(?:zes)?", r"midterm(?:s)?", r"final(?:s)?",
#     r"grade(?:s)?", r"GPA", r"credit(?:s)?",
#     r"syllabus", r"curriculum", r"timetable", r"schedule",
#     r"semester(?:s)?", r"trimester(?:s)?", r"quarter(?:s)?", r"academic year",
#     r"library", r"textbook(?:s)?", r"laborator(?:y|ies)", r"lab(?:s)?",
#     r"presentation(?:s)?", r"slides?",
#     r"scholarship(?:s)?", r"fellowship(?:s)?", r"internship(?:s)?",
#     r"diploma", r"degree(?:s)?", r"bachelor'?s?", r"master'?s?", r"phd|doctorate",
#     r"office hours", r"report card"
# ]

Extracted rows: 13601


In [ ]:
len(student_related)

13550

In [ ]:
len(student_related)

14158

In [ ]:
avg_words_per_post = students["word_count"].mean()
max_words_per_post = students["word_count"].max()
min_words_per_post = students["word_count"].min()
std_words_per_post = students["word_count"].std()
print("avg", avg_words_per_post)
print("max", max_words_per_post)
print("min", min_words_per_post)
print("std", std_words_per_post)

avg 415.54871628077166
max 1500
min 200
std 228.51769799494124


In [ ]:
student_related = student_related[["id", "gender", "age", "industry", "text", "word_count"]]
# len(student_related)

In [ ]:
# merge two tables row-wise
df = pd.concat([df, student_related], ignore_index=True)

In [ ]:
len(df)

106328

In [ ]:
df.sample(20, random_state = 42)

,id,gender,age,industry,text,word_count
61583,118791,male,33,Internet,running through cj i see that a number of web ...,302
11666,3691135,female,41,Arts,"there's this woman, who has a wild child livin...",241
3861,597870,female,33,Internet,"we all know that getting really, really drunk ...",237
77782,919128,male,26,Law & Specialized Services,"i couldn't think of a better title, besides 'm...",232
43425,947299,male,24,Law & Specialized Services,that title is a bit of a lie. if someone put a...,586
41839,3905605,female,23,Education,guest host: sara so the crew met on tuesday ni...,308
8085,3440336,female,23,Arts,this morning i posed as a teacher in a photo s...,350
72160,3825235,female,23,Education,i collect elephants. i have since my freshman ...,286
80150,670277,female,27,Education,"from: margo date: may 23, 2004 01:21 pm subjec...",554
41671,3794654,female,24,Science & Technical,funny story today. my hands are pretty soft ex...,242


In [ ]:
len(df)

106328

In [ ]:
df.to_csv("/content/drive/MyDrive/final_clean_all_data.csv", index=False)

In [ ]:
import pandas as pd
# df = pd.read_csv("/content/drive/MyDrive/final_clean_all_data.csv")

In [ ]:
len(df)

103740

In [ ]:
print(df["gender"].value_counts())
print(df["gender"].value_counts(normalize=True) * 100)

gender
male      51002
female    45197
Name: count, dtype: int64
gender
male      53.017183
female    46.982817
Name: proportion, dtype: float64


In [ ]:
print(df["age"].value_counts())

age
24    12642
23    12536
25    12446
26    10186
27     7567
17     7262
16     4903
33     4653
15     3259
34     3110
35     2687
36     2589
37     2259
14     1494
45     1118
38      996
43      951
47      857
39      784
46      739
40      692
41      652
13      553
44      448
48      428
42      388
Name: count, dtype: int64


In [ ]:
# Count texts per industry
industry_counts = df["industry"].value_counts()

print(industry_counts)

# If you prefer a DataFrame
industry_counts_df = df["industry"].value_counts().reset_index()
industry_counts_df.columns = ["industry", "count"]

industry
Student                        10695
Education                      10322
Arts                            9728
Technology                      9560
Creative Media & Culture        8327
Science & Technical             7371
Communications-Media            6760
Business & Consulting           6371
Industrial & Misc.              5901
Law & Specialized Services      5187
Internet                        4641
Non-Profit                      4629
Public Service & Governance     4067
Finance & Property              2640
Name: count, dtype: int64


In [ ]:
len(df)

96199

In [ ]:
df = df.drop_duplicates(subset=["text"])
len(df)

101484

In [ ]:
MIN_POSTS = 3  # change if needed

# filter authors with at least MIN_POSTS
cnt = df.groupby("id")["text"].transform("size")
df = df[cnt >= MIN_POSTS].copy()

# compute stats
author_counts = df.groupby("id").size()

n_authors   = df["id"].nunique()
avg_posts   = author_counts.mean()
min_posts   = author_counts.min()
max_posts   = author_counts.max()
std_posts   = author_counts.std()

print(f"authors: {n_authors}")
print(f"avg posts/author: {avg_posts:.2f}")
print(f"min posts/author: {min_posts}")
print(f"max posts/author: {max_posts}")
print(f"std of posts/author: {std_posts:.2f}")

authors: 5477
avg posts/author: 17.56
min posts/author: 3
max posts/author: 584
std of posts/author: 32.91


In [ ]:
len(df)
df.head()

,id,gender,age,industry,text,word_count
0,3581210,male,33,Finance & Property,i had an interesting conversation with my dad ...,662
1,3581210,male,33,Finance & Property,"if anything, korea is a country of extremes. e...",387
2,3581210,male,33,Finance & Property,take a read of this news article from <url> jo...,386
3,3581210,male,33,Finance & Property,"ah, the korean language...<ELONG>it looks so d...",296
4,3581210,male,33,Finance & Property,if you click on my profile you'll make a not-s...,499


In [ ]:
df.to_csv("/content/drive/MyDrive/best_clean_data.csv", index = False) # ----- after deleting duplicates and min_autor = 3
len(df)

96199

In [ ]:
# count posts per author
author_counts = df.groupby("id").size().sort_values(ascending=False)

# top 20 authors with most posts
top20 = author_counts.head(20)

print(top20)

id
1476382    584
780903     474
1596188    460
1713442    425
216413     418
883178     405
1046946    385
958176     369
2186817    363
1169141    354
2994939    327
1281160    317
1013637    296
919128     284
681976     272
1662633    271
3025353    268
1149422    247
152151     247
620124     245
dtype: int64


In [ ]:
blogtext = pd.read_csv("/content/drive/MyDrive/blogtext.csv")
len(blogtext)

681284

In [ ]:
# count posts per author
author_counts = blogtext.groupby("id").size().sort_values(ascending=False)

# top 20 authors with most posts
top20 = author_counts.head(20)

print(top20)

id
449628     4221
734562     2301
589736     2294
1975546    2261
958176     2244
1107146    2237
303162     2114
942828     2068
1270648    1951
1784456    1843
955372     1771
1078410    1731
1325355    1702
883178     1616
595404     1561
988941     1542
1093691    1533
1093457    1533
605396     1505
322624     1351
dtype: int64


In [ ]:
# df.to_csv("/content/drive/MyDrive/final_clean_all_data.csv", index = False)

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.


KeyboardInterrupt



In [ ]:
len(df)

96199

In [ ]:
import pandas as pd
dl = pd.read_csv("/content/drive/MyDrive/long_1500.csv")

In [ ]:
len(dl)

In [ ]:
# industries to drop
drop_inds = ["Student", "Technology", "Arts", "Education"]

# filter out unwanted industries
dl = dl[~dl["industry"].isin(drop_inds)].reset_index(drop=True)

industry_counts = dl["industry"].value_counts()
print(industry_counts)

industry
Creative Media & Culture       215
Communications-Media           171
Science & Technical            157
Business & Consulting          149
Internet                       141
Industrial & Misc.             114
Law & Specialized Services      91
Non-Profit                      91
Public Service & Governance     81
Finance & Property              42
Name: count, dtype: int64


In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/short_1500.csv")

In [ ]:
len(df)

89191

In [ ]:
# df.sample(10, random_state=45)
industry_counts_df = df["industry"].value_counts().reset_index()
industry_counts_df.columns = ["industry", "count"]
print(industry_counts_df)

                       industry  count
0                    Technology  10955
1                     Education  10944
2                          Arts  10596
3      Creative Media & Culture   8801
4           Science & Technical   8179
5          Communications-Media   6403
6         Business & Consulting   6205
7            Industrial & Misc.   6101
8    Law & Specialized Services   5203
9                    Non-Profit   4631
10                     Internet   4185
11  Public Service & Governance   4138
12           Finance & Property   2850


In [ ]:
finance = pd.read_csv("/content/drive/MyDrive/finance.csv")
govern = pd.read_csv("/content/drive/MyDrive/govern.csv")
internet = pd.read_csv("/content/drive/MyDrive/internet.csv")
non_profit = pd.read_csv("/content/drive/MyDrive/non_profit.csv")
law = pd.read_csv("/content/drive/MyDrive/law.csv")
industrial = pd.read_csv("/content/drive/MyDrive/industrial.csv")
business = pd.read_csv("/content/drive/MyDrive/business.csv")
media = pd.read_csv("/content/drive/MyDrive/media.csv")

In [ ]:
len(media)
# law.head()

748

In [ ]:
media = media[["id", "gender", "age", "industry", "text", "word_count"]]

In [ ]:
finance.head()

,id,gender,age,industry,text,word_count
0,3581210,male,33,Finance & Property,"just so you know, this blog isn't about being ...",678
1,3581210,male,33,Finance & Property,"no benefits, low salaries, fired for small err...",677
2,3581210,male,33,Finance & Property,"a fallen soldier? so, what is next? will there...",677
3,1795821,female,23,Finance & Property,(note: this was originally published on my old...,668
4,1795821,female,23,Finance & Property,"for a minute. 'wait a minute,' he said, 'matt ...",668


In [ ]:
# counts_df = media.groupby("id")["text"].count().reset_index(name="count")
# counts_df.sort_values("count", ascending=False).head(20)

In [ ]:
df = df[["id", "gender", "age", "industry", "text", "word_count"]]

In [ ]:
# merge two tables row-wise
df = pd.concat([df, media], ignore_index=True)

In [ ]:
df["industry"].value_counts()

,count
industry,
Technology,10955
Education,10944
Arts,10596
Creative Media & Culture,8801
Science & Technical,8179
Communications-Media,7151
Business & Consulting,6725
Industrial & Misc.,6549
Law & Specialized Services,5523


In [ ]:
df.to_csv("/content/drive/MyDrive/all_clean_data_without_students.csv", index=False)

In [ ]:
# count texts per author id
id_counts = df["id"].value_counts()

# show top 20
print(id_counts.head(20))

id
1476382    592
2186817    482
1596188    454
1713442    425
883178     405
1046946    386
216413     376
958176     369
1169141    354
2994939    331
1281160    317
1013637    296
919128     284
681976     272
1662633    271
3025353    262
1149422    247
152151     247
620124     245
1470319    244
Name: count, dtype: int64


In [ ]:
# -------------------------------------------------------------FINANCE SPLITTING

In [ ]:
dl = pd.read_csv("/content/drive/MyDrive/long_1500.csv")

In [ ]:
finance = dl[dl["industry"] == "Finance & Property"]
len(finance)

42

In [ ]:
avg_words_per_post = finance["word_count"].mean()
max_words_per_post = finance["word_count"].max()
min_words_per_post = finance["word_count"].min()
std_words_per_post = finance["word_count"].std()
print("avg", avg_words_per_post)
print("max", max_words_per_post)
print("min", min_words_per_post)
print("std", std_words_per_post)

avg 1952.1904761904761
max 3403
min 1509
std 382.0297037420403


In [ ]:
dl.head()

,id,gender,age,topic,sign,date,text,word_count,industry
0,3581210,male,33,InvestmentBanking,Aquarius,"23,July,2004","just so you know, this blog isn't about being ...",2032,Finance & Property
1,3022585,female,27,Education,Aquarius,"01,August,2004","i i don’t remember his name, but i remember th...",2018,Education
2,3667495,male,15,Science,Libra,"17,July,2004",so there they were in the cotton candy shack. ...,1948,Science & Technical
3,3568056,male,17,Sports-Recreation,Capricorn,"01,August,2004",(that means obligatory/compulsory) (which mean...,1595,Industrial & Misc.
4,589736,male,35,Technology,Aries,"05,August,2004",can't read it without a password so here's the...,5276,Technology


In [ ]:
import pandas as pd

# finance has columns: id, gender, age_bucket, industry, text, word_count

def split_into_three_chunks(text: str):
    words = str(text).split()
    n = len(words)
    base, rem = divmod(n, 3)
    sizes = [(base + 1 if i < rem else base) for i in range(3)]
    chunks, start = [], 0
    for sz in sizes:
        chunks.append(" ".join(words[start:start+sz]))
        start += sz
    return chunks  # 3 parts

new_rows = []
for _, row in finance.iterrows():
    parts = split_into_three_chunks(row["text"])
    for j, part in enumerate(parts, start=1):
        new_rows.append({
            "id": row["id"],                       # <-- unchanged
            "gender": row["gender"],
            "age": row["age"],       # use the correct column name
            "industry": row["industry"],
            "text": part,
            "word_count": len(part.split()),
            "chunk_number": j,                     # 1..3 (traceability)
            "uid": f"{row['id']}_p{j}"             # optional unique key per chunk
        })

new_finance = pd.DataFrame(new_rows)[
    ["id", "gender", "age", "industry", "text", "word_count", "chunk_number", "uid"]
]

# sanity checks
print(f"Original rows: {len(finance)}  →  After split: {len(new_finance)} (x3 expected)")
print(new_finance.groupby("id")["chunk_number"].nunique().value_counts().sort_index())
print(new_finance["word_count"].describe())


Original rows: 42  →  After split: 126 (x3 expected)
chunk_number
3    15
Name: count, dtype: int64
count     126.000000
mean      650.730159
std       126.320951
min       503.000000
25%       565.250000
50%       606.000000
75%       718.000000
max      1135.000000
Name: word_count, dtype: float64


In [ ]:
new_finance.head()

,id,gender,age,industry,text,word_count,chunk_number,uid
0,3581210,male,33,Finance & Property,"just so you know, this blog isn't about being ...",678,1,3581210_p1
1,3581210,male,33,Finance & Property,"no benefits, low salaries, fired for small err...",677,2,3581210_p2
2,3581210,male,33,Finance & Property,"a fallen soldier? so, what is next? will there...",677,3,3581210_p3
3,1795821,female,23,Finance & Property,(note: this was originally published on my old...,668,1,1795821_p1
4,1795821,female,23,Finance & Property,"for a minute. 'wait a minute,' he said, 'matt ...",668,2,1795821_p2


In [ ]:
finance = new_finance.copy()
finance.to_csv("/content/drive/MyDrive/finance.csv")

In [ ]:
# ------------------------------------ Public service & governence splitting

In [ ]:
govern = dl[dl["industry"] == "Public Service & Governance"]
len(govern)

81

In [ ]:
avg_words_per_post = govern["word_count"].mean()
max_words_per_post = govern["word_count"].max()
min_words_per_post = govern["word_count"].min()
std_words_per_post = govern["word_count"].std()
print("avg", avg_words_per_post)
print("max", max_words_per_post)
print("min", min_words_per_post)
print("std", std_words_per_post)

avg 2475.1234567901233
max 14979
min 1509
std 1715.319433682223


In [ ]:
import pandas as pd
import math
import re

# --- INPUT: split texts in `govern` only ---
# columns: id, gender, age_bucket, industry, text, word_count
df = govern.copy()

# We DO NOT collapse/merge IDs. We treat the 146 rows as originals.

def parts_by_rule(L: int) -> int:
    """Rule:
       <10k -> first digit = n -> 2*n parts
       >=10k -> first two digits = n -> 2*n parts
       Then cap so every chunk has >200 words.
    """
    s = str(L)
    n_pref = 2 * (int(s[:2]) if L >= 10000 else int(s[0]))

    # guarantee strictly > 200 words per chunk
    max_parts_allowed = max(1, L // 201)   # 201 => every chunk >=201
    return max(1, min(n_pref, max_parts_allowed))

def split_equal(text: str, n: int):
    words = str(text).split()
    L = len(words)
    if L == 0 or n <= 1:
        return [str(text)]
    base, rem = divmod(L, n)
    sizes = [(base + 1 if i < rem else base) for i in range(n)]
    out, i = [], 0
    for sz in sizes:
        out.append(" ".join(words[i:i+sz]))
        i += sz
    return out

rows = []
for _, r in df.iterrows():
    words_len = len(str(r["text"]).split())
    n_parts = parts_by_rule(words_len)
    parts = split_equal(r["text"], n_parts)

    for part in parts:
        rows.append({
            "id": r["id"],   # <-- keep SAME id (no suffix)
            "gender": r["gender"],
            "age": r["age"],
            "industry": r["industry"],
            "text": part,
            "word_count": len(part.split())
        })

new_govern = pd.DataFrame(rows, columns=["id","gender","age","industry","text","word_count"])

# --- sanity: counts and minimum length (>200) ---
print("Original govern rows:", len(df))                     # should be 146
print("New govern rows:", len(new_govern))
min_wc = new_govern["word_count"].min()
print("Min chunk word_count:", min_wc)
if min_wc <= 200:
    print("WARNING: A chunk ≤200 words slipped through. Investigate the source row.")
else:
    print("All chunks > 200 words ✅")

# longest chunks preview
print(new_govern.sort_values("word_count", ascending=False).head(10)[["id","word_count"]])


Original govern rows: 81
New govern rows: 312
Min chunk word_count: 502
All chunks > 200 words ✅
          id  word_count
0    3857088         991
34   3486137         991
35   3486137         991
1    3857088         990
270  4208262         983
271  4208262         982
43   2586414         977
27   3991685         977
10   3442766         977
42   2586414         977


In [ ]:
new_govern = new_govern[new_govern["word_count"] >= 200]

govern = new_govern.copy()
govern.to_csv("/content/drive/MyDrive/govern.csv")

In [ ]:
# ---------------------------------------- INTERNET SPLITTING---------------------

In [ ]:
internet = dl[dl["industry"] == "Internet"]
len(internet)

141

In [ ]:
avg_words_per_post = internet["word_count"].mean()
max_words_per_post = internet["word_count"].max()
min_words_per_post = internet["word_count"].min()
std_words_per_post = internet["word_count"].std()
print("avg", avg_words_per_post)
print("max", max_words_per_post)
print("min", min_words_per_post)
print("std", std_words_per_post)

avg 2926.241134751773
max 15469
min 1508
std 2132.091065800726


In [ ]:
print(internet["word_count"].sort_values(ascending=False))

1110    15469
1108    13397
1099     9851
1085     9695
1092     8189
        ...  
541      1537
686      1524
282      1519
379      1512
623      1508
Name: word_count, Length: 141, dtype: int64


In [ ]:
import pandas as pd
import math
import re

# --- INPUT: split texts in `internet` only ---
# columns: id, gender, age_bucket, industry, text, word_count
df = internet.copy()

# We DO NOT collapse/merge IDs. We treat the rows as originals.

def parts_by_rule(L: int) -> int:
    s = str(L)
    n_pref = 2 * (int(s[:2]) if L >= 10000 else int(s[0]))
    max_parts_allowed = max(1, L // 200)   # ensure each chunk ≥200 words
    return max(1, min(n_pref, max_parts_allowed))

def split_equal(text: str, n: int):
    words = str(text).split()
    L = len(words)
    if L == 0 or n <= 1:
        return [str(text)]
    base, rem = divmod(L, n)
    sizes = [(base + 1 if i < rem else base) for i in range(n)]
    out, i = [], 0
    for sz in sizes:
        out.append(" ".join(words[i:i+sz]))
        i += sz
    return out

rows = []
for _, r in df.iterrows():
    words_len = len(str(r["text"]).split())
    n_parts = parts_by_rule(words_len)
    parts = split_equal(r["text"], n_parts)

    for part in parts:
        rows.append({
            "id": r["id"],   # <-- keep SAME id (no suffix)
            "gender": r["gender"],
            "age": r["age"],
            "industry": r["industry"],
            "text": part,
            "word_count": len(part.split())
        })

new_internet = pd.DataFrame(rows, columns=["id","gender","age","industry","text","word_count"])

# --- sanity: counts and minimum length (>200) ---
print("Original internet rows:", len(df))
print("New internet rows:", len(new_internet))
min_wc = new_internet["word_count"].min()
print("Min chunk word_count:", min_wc)
if min_wc <= 200:
    print("WARNING: A chunk ≤200 words slipped through. Investigate the source row.")
else:
    print("All chunks > 200 words ✅")

# longest chunks preview
print(new_internet.sort_values("word_count", ascending=False).head(20)[["id","word_count"]])

# Diagnose expected parts per row BEFORE splitting
tmp = internet.copy()
tmp["L"] = tmp["text"].astype(str).str.split().str.len()
tmp["n_pref"] = tmp["L"].astype(str).str[:2].where(tmp["L"]>=10000, tmp["L"].astype(str).str[0]).astype(int) * 2
tmp["max_parts_allowed"] = (tmp["L"] // 200).clip(lower=1)   # ≥200 words per chunk
tmp["n_parts"] = tmp[["n_pref","max_parts_allowed"]].min(axis=1)

print("Original rows:", len(tmp))
print("Expected rows after split:", int(tmp["n_parts"].sum()))
print("\nDistribution of n_parts:")
print(tmp["n_parts"].value_counts().sort_index())

Original internet rows: 141
New internet rows: 664
Min chunk word_count: 500
All chunks > 200 words ✅
          id  word_count
24   1960271         984
25   1960271         983
660  3347383         973
661  3347383         972
254  3568204         961
255  3568204         960
151  3727643         953
150  3727643         953
33   4209930         952
32   4209930         952
446  2994939         946
216  1510754         946
217  1510754         945
310  3447453         945
447  2994939         945
311  3447453         945
86   4196565         938
87   4196565         937
230  1510754         924
231  1510754         923
Original rows: 141
Expected rows after split: 664

Distribution of n_parts:
n_parts
2     65
4     33
6     23
8      4
10     5
12     5
14     1
16     1
18     2
26     1
30     1
Name: count, dtype: int64


In [ ]:
len(new_internet)

664

In [ ]:
len(new_internet[new_internet["word_count"] <= 100])
new_internet = new_internet[new_internet["word_count"] >= 200]

internet = new_internet.copy()
internet.to_csv("/content/drive/MyDrive/internet.csv")

In [ ]:
# ------------------------------- Non Profit -----------------------------

In [ ]:
non_profit = dl[dl["industry"] == "Non-Profit"]
len(non_profit)

91

In [ ]:
avg_words_per_post = non_profit["word_count"].mean()
max_words_per_post = non_profit["word_count"].max()
min_words_per_post = non_profit["word_count"].min()
std_words_per_post = non_profit["word_count"].std()
print("avg", avg_words_per_post)
print("max", max_words_per_post)
print("min", min_words_per_post)
print("std", std_words_per_post)

avg 2754.4505494505493
max 20847
min 1503
std 2639.9655059351576


In [ ]:
import pandas as pd
import math
import re

# --- INPUT: split texts in `internet` only ---
# columns: id, gender, age_bucket, industry, text, word_count
df = non_profit.copy()

# We DO NOT collapse/merge IDs. We treat the rows as originals.

def parts_by_rule(L: int) -> int:
    s = str(L)
    n_pref = 2 * (int(s[:2]) if L >= 10000 else int(s[0]))
    max_parts_allowed = max(1, L // 200)   # ensure each chunk ≥200 words
    return max(1, min(n_pref, max_parts_allowed))

def split_equal(text: str, n: int):
    words = str(text).split()
    L = len(words)
    if L == 0 or n <= 1:
        return [str(text)]
    base, rem = divmod(L, n)
    sizes = [(base + 1 if i < rem else base) for i in range(n)]
    out, i = [], 0
    for sz in sizes:
        out.append(" ".join(words[i:i+sz]))
        i += sz
    return out

rows = []
for _, r in df.iterrows():
    words_len = len(str(r["text"]).split())
    n_parts = parts_by_rule(words_len)
    parts = split_equal(r["text"], n_parts)

    for part in parts:
        rows.append({
            "id": r["id"],   # <-- keep SAME id (no suffix)
            "gender": r["gender"],
            "age": r["age"],
            "industry": r["industry"],
            "text": part,
            "word_count": len(part.split())
        })

new_non_profit = pd.DataFrame(rows, columns=["id","gender","age","industry","text","word_count"])

# --- sanity: counts and minimum length (>200) ---
print("Original internet rows:", len(df))
print("New internet rows:", len(new_internet))
min_wc = new_internet["word_count"].min()
print("Min chunk word_count:", min_wc)
if min_wc <= 200:
    print("WARNING: A chunk ≤200 words slipped through. Investigate the source row.")
else:
    print("All chunks > 200 words ✅")

# longest chunks preview
print(new_internet.sort_values("word_count", ascending=False).head(20)[["id","word_count"]])

# Diagnose expected parts per row BEFORE splitting
tmp = non_profit.copy()
tmp["L"] = tmp["text"].astype(str).str.split().str.len()
tmp["n_pref"] = tmp["L"].astype(str).str[:2].where(tmp["L"]>=10000, tmp["L"].astype(str).str[0]).astype(int) * 2
tmp["max_parts_allowed"] = (tmp["L"] // 200).clip(lower=1)   # ≥200 words per chunk
tmp["n_parts"] = tmp[["n_pref","max_parts_allowed"]].min(axis=1)

print("Original rows:", len(tmp))
print("Expected rows after split:", int(tmp["n_parts"].sum()))

Original internet rows: 91
New internet rows: 664
Min chunk word_count: 500
All chunks > 200 words ✅
          id  word_count
24   1960271         984
25   1960271         983
660  3347383         973
661  3347383         972
254  3568204         961
255  3568204         960
151  3727643         953
150  3727643         953
33   4209930         952
32   4209930         952
446  2994939         946
216  1510754         946
217  1510754         945
310  3447453         945
447  2994939         945
311  3447453         945
86   4196565         938
87   4196565         937
230  1510754         924
231  1510754         923
Original rows: 91
Expected rows after split: 398


In [ ]:
len(new_non_profit)

new_internet = new_internet[new_internet["word_count"] >= 200]

In [ ]:
non_profit = new_non_profit.copy()
non_profit.to_csv("/content/drive/MyDrive/non_profit.csv")

In [ ]:
# ---------------------------------- Law & Specialized Services --------------------

In [ ]:
law = dl[dl["industry"] == "Law & Specialized Services"]
len(law)

91

In [ ]:
avg_words_per_post = law["word_count"].mean()
max_words_per_post = law["word_count"].max()
min_words_per_post = law["word_count"].min()
std_words_per_post = law["word_count"].std()
print("avg", avg_words_per_post)
print("max", max_words_per_post)
print("min", min_words_per_post)
print("std", std_words_per_post)

avg 2297.1098901098903
max 14554
min 1514
std 1461.2095936856138


In [ ]:
import pandas as pd
import math
import re

# --- INPUT: split texts in `internet` only ---
# columns: id, gender, age_bucket, industry, text, word_count
df = law.copy()

# We DO NOT collapse/merge IDs. We treat the rows as originals.

def parts_by_rule(L: int) -> int:
    s = str(L)
    n_pref = 2 * (int(s[:2]) if L >= 10000 else int(s[0]))
    max_parts_allowed = max(1, L // 200)   # ensure each chunk ≥200 words
    return max(1, min(n_pref, max_parts_allowed))

def split_equal(text: str, n: int):
    words = str(text).split()
    L = len(words)
    if L == 0 or n <= 1:
        return [str(text)]
    base, rem = divmod(L, n)
    sizes = [(base + 1 if i < rem else base) for i in range(n)]
    out, i = [], 0
    for sz in sizes:
        out.append(" ".join(words[i:i+sz]))
        i += sz
    return out

rows = []
for _, r in df.iterrows():
    words_len = len(str(r["text"]).split())
    n_parts = parts_by_rule(words_len)
    parts = split_equal(r["text"], n_parts)

    for part in parts:
        rows.append({
            "id": r["id"],   # <-- keep SAME id (no suffix)
            "gender": r["gender"],
            "age": r["age"],
            "industry": r["industry"],
            "text": part,
            "word_count": len(part.split())
        })

new_law = pd.DataFrame(rows, columns=["id","gender","age","industry","text","word_count"])

# --- sanity: counts and minimum length (>200) ---
print("Original internet rows:", len(df))
print("New internet rows:", len(new_law))
min_wc = new_law["word_count"].min()
print("Min chunk word_count:", min_wc)
if min_wc <= 200:
    print("WARNING: A chunk ≤200 words slipped through. Investigate the source row.")
else:
    print("All chunks > 200 words ✅")

# longest chunks preview
print(new_law.sort_values("word_count", ascending=False).head(20)[["id","word_count"]])

# Diagnose expected parts per row BEFORE splitting
tmp = law.copy()
tmp["L"] = tmp["text"].astype(str).str.split().str.len()
tmp["n_pref"] = tmp["L"].astype(str).str[:2].where(tmp["L"]>=10000, tmp["L"].astype(str).str[0]).astype(int) * 2
tmp["max_parts_allowed"] = (tmp["L"] // 200).clip(lower=1)   # ≥200 words per chunk
tmp["n_parts"] = tmp[["n_pref","max_parts_allowed"]].min(axis=1)

print("Original rows:", len(tmp))
print("Expected rows after split:", int(tmp["n_parts"].sum()))

Original internet rows: 91
New internet rows: 320
Min chunk word_count: 501
All chunks > 200 words ✅
          id  word_count
130   216413         967
131   216413         967
44   4195534         966
45   4195534         966
137   860293         963
136   860293         963
312  3473351         958
313  3473351         957
308  3473351         952
309  3473351         951
307  3670239         943
306  3670239         943
102   216413         940
103   216413         939
53   1919658         930
52   1919658         930
112   216413         928
113   216413         928
143  3360798         921
142  3360798         921
Original rows: 91
Expected rows after split: 320


In [ ]:
len(new_law)

320

In [ ]:
new_law = new_law[new_law["word_count"] >= 200]

law = new_law.copy()
law.to_csv("/content/drive/MyDrive/law.csv")

In [ ]:
# ---------------------------------------- Business & Consulting

In [ ]:
business = dl[dl["industry"] == "Business & Consulting"]
len(business)

149

In [ ]:
avg_words_per_post = business["word_count"].mean()
max_words_per_post = business["word_count"].max()
min_words_per_post = business["word_count"].min()
std_words_per_post = business["word_count"].std()
print("avg", avg_words_per_post)
print("max", max_words_per_post)
print("min", min_words_per_post)
print("std", std_words_per_post)

avg 2299.8389261744965
max 17191
min 1501
std 1489.2298801870998


In [ ]:
import pandas as pd
import math
import re

# --- INPUT: split texts in `internet` only ---
# columns: id, gender, age_bucket, industry, text, word_count
df = business.copy()

# We DO NOT collapse/merge IDs. We treat the rows as originals.

def parts_by_rule(L: int) -> int:
    s = str(L)
    n_pref = 2 * (int(s[:2]) if L >= 10000 else int(s[0]))
    max_parts_allowed = max(1, L // 200)   # ensure each chunk ≥200 words
    return max(1, min(n_pref, max_parts_allowed))

def split_equal(text: str, n: int):
    words = str(text).split()
    L = len(words)
    if L == 0 or n <= 1:
        return [str(text)]
    base, rem = divmod(L, n)
    sizes = [(base + 1 if i < rem else base) for i in range(n)]
    out, i = [], 0
    for sz in sizes:
        out.append(" ".join(words[i:i+sz]))
        i += sz
    return out

rows = []
for _, r in df.iterrows():
    words_len = len(str(r["text"]).split())
    n_parts = parts_by_rule(words_len)
    parts = split_equal(r["text"], n_parts)

    for part in parts:
        rows.append({
            "id": r["id"],   # <-- keep SAME id (no suffix)
            "gender": r["gender"],
            "age": r["age"],
            "industry": r["industry"],
            "text": part,
            "word_count": len(part.split())
        })

new_business = pd.DataFrame(rows, columns=["id","gender","age","industry","text","word_count"])

# --- sanity: counts and minimum length (>200) ---
print("Original internet rows:", len(df))
print("New internet rows:", len(new_business))
min_wc = new_business["word_count"].min()
print("Min chunk word_count:", min_wc)
if min_wc <= 200:
    print("WARNING: A chunk ≤200 words slipped through. Investigate the source row.")
else:
    print("All chunks > 200 words ✅")

# longest chunks preview
print(new_business.sort_values("word_count", ascending=False).head(20)[["id","word_count"]])

# Diagnose expected parts per row BEFORE splitting
tmp = business.copy()
tmp["L"] = tmp["text"].astype(str).str.split().str.len()
tmp["n_pref"] = tmp["L"].astype(str).str[:2].where(tmp["L"]>=10000, tmp["L"].astype(str).str[0]).astype(int) * 2
tmp["max_parts_allowed"] = (tmp["L"] // 200).clip(lower=1)   # ≥200 words per chunk
tmp["n_parts"] = tmp[["n_pref","max_parts_allowed"]].min(axis=1)

print("Original rows:", len(tmp))
print("Expected rows after split:", int(tmp["n_parts"].sum()))

Original internet rows: 149
New internet rows: 520
Min chunk word_count: 501
All chunks > 200 words ✅
          id  word_count
300  3734929         997
301  3734929         996
464  3247172         991
430  1604072         991
431  1604072         991
362  3361739         990
465  3247172         990
363  3361739         990
432  1604072         989
433  1604072         989
356   993945         988
296  3010250         987
357   993945         987
297  3010250         986
298  3734929         978
299  3734929         978
245  2268074         965
244  2268074         965
188  3637189         956
189  3637189         955
Original rows: 149
Expected rows after split: 520


In [ ]:
len(new_business)

520

In [ ]:
new_business = new_business[new_business["word_count"] >= 200]

business = new_business.copy()
business.to_csv("/content/drive/MyDrive/business.csv")

In [ ]:
# ------------------------------------ Industrial & Misc.

In [ ]:
industrial = dl[dl["industry"] == "Industrial & Misc."]
len(industrial)

114

In [ ]:
avg_words_per_post = industrial["word_count"].mean()
max_words_per_post = industrial["word_count"].max()
min_words_per_post = industrial["word_count"].min()
std_words_per_post = industrial["word_count"].std()
print("avg", avg_words_per_post)
print("max", max_words_per_post)
print("min", min_words_per_post)
print("std", std_words_per_post)

avg 2541.3333333333335
max 22991
min 1504
std 2719.5302713437786


In [ ]:
import pandas as pd
import math
import re

# --- INPUT: split texts in `internet` only ---
# columns: id, gender, age_bucket, industry, text, word_count
df = industrial.copy()

# We DO NOT collapse/merge IDs. We treat the rows as originals.

def parts_by_rule(L: int) -> int:
    s = str(L)
    n_pref = 2 * (int(s[:2]) if L >= 10000 else int(s[0]))
    max_parts_allowed = max(1, L // 200)   # ensure each chunk ≥200 words
    return max(1, min(n_pref, max_parts_allowed))

def split_equal(text: str, n: int):
    words = str(text).split()
    L = len(words)
    if L == 0 or n <= 1:
        return [str(text)]
    base, rem = divmod(L, n)
    sizes = [(base + 1 if i < rem else base) for i in range(n)]
    out, i = [], 0
    for sz in sizes:
        out.append(" ".join(words[i:i+sz]))
        i += sz
    return out

rows = []
for _, r in df.iterrows():
    words_len = len(str(r["text"]).split())
    n_parts = parts_by_rule(words_len)
    parts = split_equal(r["text"], n_parts)

    for part in parts:
        rows.append({
            "id": r["id"],   # <-- keep SAME id (no suffix)
            "gender": r["gender"],
            "age": r["age"],
            "industry": r["industry"],
            "text": part,
            "word_count": len(part.split())
        })

new_industrial = pd.DataFrame(rows, columns=["id","gender","age","industry","text","word_count"])

# --- sanity: counts and minimum length (>200) ---
print("Original internet rows:", len(df))
print("New internet rows:", len(new_industrial))
min_wc = new_industrial["word_count"].min()
print("Min chunk word_count:", min_wc)
if min_wc <= 200:
    print("WARNING: A chunk ≤200 words slipped through. Investigate the source row.")
else:
    print("All chunks > 200 words ✅")

# longest chunks preview
print(new_industrial.sort_values("word_count", ascending=False).head(20)[["id","word_count"]])

# Diagnose expected parts per row BEFORE splitting
tmp = industrial.copy()
tmp["L"] = tmp["text"].astype(str).str.split().str.len()
tmp["n_pref"] = tmp["L"].astype(str).str[:2].where(tmp["L"]>=10000, tmp["L"].astype(str).str[0]).astype(int) * 2
tmp["max_parts_allowed"] = (tmp["L"] // 200).clip(lower=1)   # ≥200 words per chunk
tmp["n_parts"] = tmp[["n_pref","max_parts_allowed"]].min(axis=1)

print("Original rows:", len(tmp))
print("Expected rows after split:", int(tmp["n_parts"].sum()))

Original internet rows: 114
New internet rows: 448
Min chunk word_count: 506
All chunks > 200 words ✅
          id  word_count
447  3621730         986
446  3621730         986
90   2479099         973
91   2479099         972
352  2947297         968
353  2947297         967
36   3246789         966
37   3246789         965
227  3599480         961
226  3599480         961
257  3898743         932
256  3898743         932
294  3372901         927
112  2479099         927
82   2479099         926
113  2479099         926
295  3372901         926
83   2479099         925
99   2479099         923
98   2479099         923
Original rows: 114
Expected rows after split: 448


In [ ]:
# Count how many chunks each original id produced
chunk_counts = new_industrial.groupby("id").size().sort_values(ascending=False)

# Show top 10 ids with most chunks
print(chunk_counts.head(10))

id
2479099    46
3543879    44
3507038    38
2508445    32
3246789    20
3504961    16
3342597    16
2181784    12
3431994    12
4222288    10
dtype: int64


In [ ]:
new_industrial = new_industrial[new_industrial["word_count"] >= 200]

industrial = new_industrial.copy()
industrial.to_csv("/content/drive/MyDrive/industrial.csv")

In [ ]:
# ------------------------ Communications-Media ----------------------------

In [ ]:
media = dl[dl["industry"] == "Communications-Media"]
len(media)

171

In [ ]:
avg_words_per_post = media["word_count"].mean()
max_words_per_post = media["word_count"].max()
min_words_per_post = media["word_count"].min()
std_words_per_post = media["word_count"].std()
print("avg", avg_words_per_post)
print("max", max_words_per_post)
print("min", min_words_per_post)
print("std", std_words_per_post)

avg 2727.3040935672516
max 55333
min 1504
std 4316.579382544316


In [ ]:
print(media.sort_values("word_count", ascending=False).head(10)[["id","word_count"]])

           id  word_count
616   3414073       55333
1125  3380464       15737
1126  3380464        9702
1142  3418388        6613
475   3630293        5720
1127  3380464        5473
1122  3380464        5424
389    619960        5297
1130  3380464        5187
957   3303228        4809


In [ ]:
import pandas as pd
import math
import re

# --- INPUT: split texts in `internet` only ---
# columns: id, gender, age_bucket, industry, text, word_count
df = media.copy()

# We DO NOT collapse/merge IDs. We treat the rows as originals.

def parts_by_rule(L: int) -> int:
    s = str(L)
    n_pref = 2 * (int(s[:2]) if L >= 10000 else int(s[0]))
    max_parts_allowed = max(1, L // 200)   # ensure each chunk ≥200 words
    return max(1, min(n_pref, max_parts_allowed))

def split_equal(text: str, n: int):
    words = str(text).split()
    L = len(words)
    if L == 0 or n <= 1:
        return [str(text)]
    base, rem = divmod(L, n)
    sizes = [(base + 1 if i < rem else base) for i in range(n)]
    out, i = [], 0
    for sz in sizes:
        out.append(" ".join(words[i:i+sz]))
        i += sz
    return out

rows = []
for _, r in df.iterrows():
    words_len = len(str(r["text"]).split())
    n_parts = parts_by_rule(words_len)
    parts = split_equal(r["text"], n_parts)

    for part in parts:
        rows.append({
            "id": r["id"],   # <-- keep SAME id (no suffix)
            "gender": r["gender"],
            "age": r["age"],
            "industry": r["industry"],
            "text": part,
            "word_count": len(part.split())
        })

new_media = pd.DataFrame(rows, columns=["id","gender","age","industry","text","word_count"])

# --- sanity: counts and minimum length (>200) ---
print("Original internet rows:", len(df))
print("New internet rows:", len(new_media))
min_wc = new_media["word_count"].min()
print("Min chunk word_count:", min_wc)
if min_wc <= 200:
    print("WARNING: A chunk ≤200 words slipped through. Investigate the source row.")
else:
    print("All chunks > 200 words ✅")

# longest chunks preview
print(new_media.sort_values("word_count", ascending=False).head(20)[["id","word_count"]])

# Diagnose expected parts per row BEFORE splitting
tmp = media.copy()
tmp["L"] = tmp["text"].astype(str).str.split().str.len()
tmp["n_pref"] = tmp["L"].astype(str).str[:2].where(tmp["L"]>=10000, tmp["L"].astype(str).str[0]).astype(int) * 2
tmp["max_parts_allowed"] = (tmp["L"] // 200).clip(lower=1)   # ≥200 words per chunk
tmp["n_parts"] = tmp[["n_pref","max_parts_allowed"]].min(axis=1)

print("Original rows:", len(tmp))
print("Expected rows after split:", int(tmp["n_parts"].sum()))

Original internet rows: 171
New internet rows: 748
Min chunk word_count: 500
All chunks > 200 words ✅
          id  word_count
723  2162182         999
722  2162182         999
458  2713315         998
459  2713315         998
89   3447484         994
88   3447484         994
536  4320066         990
537  4320066         989
464  2892926         983
465  2892926         983
672  4220690         978
673  4220690         977
538  3380464         977
539  3380464         976
468  3303228         974
469  3303228         974
224  3454948         969
225  3454948         969
68   3447484         959
69   3447484         958
Original rows: 171
Expected rows after split: 748


In [ ]:
len(new_media)

748

In [ ]:
new_media = new_media[new_media["word_count"] >= 200]

media = new_media.copy()
media.to_csv("/content/drive/MyDrive/media.csv")

In [ ]:
MIN_POSTS = 3  # change to what you want (e.g., 5)

# assume you have an 'author_id' column
cnt = df.groupby("id")["text"].transform("size")
df = df[cnt >= MIN_POSTS].copy()

# how many authors remain / avg posts per author
n_authors = df["id"].nunique()
avg_posts = (df.groupby("id").size().mean())
print(f"authors: {n_authors}, avg posts/author: {avg_posts:.2f}")

In [ ]:
len(df)

205401

In [ ]:
df.head()

,Unnamed: 0,id,gender,age,topic,sign,date,text,age_bucket,word_count,industry,post_clean
5,5,3581210,male,33,InvestmentBanking,Aquarius,"10,June,2004",I had an interesting conversation with my Dad ...,30s,662,Finance & Property,I had an interesting conversation with my Dad ...
6,6,3581210,male,33,InvestmentBanking,Aquarius,"10,June,2004",Somehow Coca-Cola has a way of summing up thin...,30s,196,Finance & Property,Somehow Coca-Cola has a way of summing up thin...
7,7,3581210,male,33,InvestmentBanking,Aquarius,"10,June,2004","If anything, Korea is a country of extremes. ...",30s,387,Finance & Property,"If anything, Korea is a country of extremes. ..."
8,8,3581210,male,33,InvestmentBanking,Aquarius,"10,June,2004",Take a read of this news article from urlLink ...,30s,386,Finance & Property,Take a read of this news article from urlLink ...
9,9,3581210,male,33,InvestmentBanking,Aquarius,"09,June,2004",I surf the English news sites a lot looking fo...,30s,160,Finance & Property,I surf the English news sites a lot looking fo...


In [ ]:
# ensure stable ids to re-merge later
if "post_id" not in df.columns:
    df = df.reset_index(drop=False).rename(columns={"index": "post_id"})

# (re)compute word_count if needed
df["word_count"] = df["text"].str.split().str.len()

# masks
keep_mask  = (df["word_count"] >= 100) & (df["word_count"] <= 1200)
long_mask  = df["word_count"] > 1200

# split
df_kept  = df[keep_mask].copy()
df_long  = df[long_mask].copy()

# save for later balancing
df_kept.to_csv("bac_clean_100_1200.csv", index=False)
df_long.to_csv("bac_long_over_1200.csv", index=False)

# continue with cleaned set for training
df = df_kept

In [ ]:
len(df)

205401

In [ ]:
long = pd.read_csv("/content/drive/MyDrive/long_over_1200.csv")
len(long)

4647

In [ ]:
print("posts:", len(df))
print("authors:", df["id"].nunique())
print(df.groupby("id").size().describe())  # per-author counts
df["word_count"].describe(percentiles=[.5,.9,.95,.99])


posts: 205572
authors: 9530
count    9530.000000
mean       21.571039
std        43.800572
min         1.000000
25%         4.000000
50%         9.000000
75%        20.000000
max       973.000000
dtype: float64


,word_count
count,205572.000000
mean,308.861027
std,207.808276
min,101.000000
50%,241.000000
90%,599.000000
95%,761.000000
99%,1045.000000
max,1200.000000


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# save into your Drive (e.g. My Drive/mtap_data/)
df_kept.to_csv("/content/drive/My Drive/clean_100_1200.csv", index=False)
df_long.to_csv("/content/drive/My Drive/long_over_1200.csv", index=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
